# 🔬 파라미터 실험 — temperature · max_new_tokens · top_p

---

## 🎯 오늘의 상황

> **법률 AI 스타트업**에 개발자로 입사했습니다.  
> 팀장이 말합니다: "EXAONE으로 법률 Q&A 챗봇을 만드는데, 파라미터를 어떻게 설정해야 할지 모르겠어요.  
> 같은 질문인데 매번 다른 답이 나와요. 이게 버그인가요?"
>  
> 오늘은 파라미터를 **직접 실험**하며 이 질문에 답해봅니다.

---

## 📋 실험 순서

```
실험 1-A. temperature 4단계 비교
실험 1-B. temperature=0.0 반복 3회 (동일한가?)
실험 1-C. max_new_tokens에 따른 응답 변화
실험 1-D. top_p 3단계 비교
실험 1-E. 환각(Hallucination) 관찰
```

---

## 💡 핵심 개념 — 파라미터 역할 구분

| 파라미터 | 역할 | 비유 |
|---------|------|------|
| `temperature` | 응답 다양성 조절 | 식당 선택 폭 (단골집 vs 눈감고 들어가기) |
| `max_new_tokens` | 새로 생성할 토큰 상한선 | "이 이상은 말하지 마" |
| `top_p` | 단어 후보 범위 제한 | 주사위 눈 수 줄이기 |

> ⚠️ **실험의 원칙: 한 번에 하나씩만 바꾼다!**  
> 여러 개를 동시에 바꾸면 무엇 때문에 달라졌는지 알 수 없습니다.

> 💡 **OpenAI `max_tokens` vs HuggingFace `max_new_tokens`**  
> OpenAI는 입력+출력 합산이지만, EXAONE은 새로 생성하는 토큰만 셉니다.

---
## 🔧 사전 준비

In [ ]:
# 패키지 설치 (처음 실행 시만)
!pip install -q --upgrade transformers accelerate

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

print("📥 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

print("📥 모델 로드 중... (약 2~3분 소요)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto",
)
print(f"✅ 모델 로드 완료! GPU 메모리: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
# generate_response 함수 + 토큰 추적기 (Day 1에서 만든 것)
def generate_response(messages, max_new_tokens=256, temperature=0.7, top_p=0.9):
    """EXAONE 로컬 추론 — 메시지 리스트를 받아 응답 dict를 반환한다."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    input_ids = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).input_ids.to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True if temperature > 0 else False,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][input_ids.shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {
        "content": answer,
        "input_tokens": input_ids.shape[1],
        "output_tokens": len(generated_ids),
        "finish_reason": "stop" if len(generated_ids) < max_new_tokens else "length",
    }


class TokenTracker:
    def __init__(self):
        self.total_input  = 0
        self.total_output = 0
        self.call_count   = 0

    def add(self, result):
        self.total_input  += result["input_tokens"]
        self.total_output += result["output_tokens"]
        self.call_count   += 1

    def report(self):
        api_cost = (self.total_input * 0.15 + self.total_output * 0.60) / 1_000_000
        print(f"총 {self.call_count}회 호출 | 입력 {self.total_input:,} + 출력 {self.total_output:,} 토큰")
        print(f"로컬 추론 비용: ₩0")
        print(f"(참고) API 환산 비용: ${api_cost:.4f} (약 {api_cost * 1400:.1f}원)")


tracker = TokenTracker()
print("✅ 준비 완료!")

---
## 실험 전 예측 ✍️

실험 **전에** 결과를 예측해보세요. 예측과 실제 결과를 비교하면 기억에 더 오래 남습니다.

| 질문 | 내 예측 |
|------|--------|
| temperature=0.0 같은 질문 3번 → 결과가 같을까, 다를까? | |
| temperature=1.5에서 어떤 이상한 점이 생길까? | |
| 법률 Q&A에 가장 적합한 temperature는? | |
| max_new_tokens=30이면 어떤 일이 생길까? | |

---
## 실험 1-A. temperature 4단계 비교

같은 질문을 temperature만 바꿔가며 4번 호출합니다.  
응답이 어떻게 달라지는지 관찰하세요.

| temperature | 특성 | 어울리는 상황 |
|------------|------|---------------|
| `0.0` | 항상 같은 답 (그리디 디코딩) | 데이터 추출, 분류 |
| `0.5` | 약간의 변동 | **법률·의료 Q&A** |
| `1.0` | 창의적·다양 | 글쓰기, 아이디어 |
| `1.5` | 무작위 (환각 위험) | 실험적 창작 |

> 💡 EXAONE에서 `temperature=0.0`은 `do_sample=False` (그리디 디코딩)로 처리됩니다.

In [ ]:
# 모든 실험에서 질문은 이 변수를 사용합니다 (고정)
question = "법원 판결문 작성 시 가장 중요한 요소는?"

# temperature를 0.0 → 0.5 → 1.0 → 1.5 순서로 바꿔가며 호출
for temp in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n{'='*50}")
    print(f"[temperature = {temp}]")
    print(f"{'='*50}")

    result = generate_response(
        messages=[{"role": "user", "content": question}],
        temperature=temp,       # ← 이 값만 바뀝니다
        max_new_tokens=100,     # 고정
        top_p=0.9,              # 고정
    )

    print(result["content"])
    print(f"\n  → 종료 이유: {result['finish_reason']}")
    print(f"  → 출력 토큰: {result['output_tokens']}")

    tracker.add(result)

**📝 실험 1-A 기록표 — 직접 채워보세요**

| temperature | 응답이 일관적인가? | 어휘가 다양한가? | 법률 Q&A 적합도 | 특이사항 |
|------------|-----------------|----------------|----------------|----------|
| `0.0` | | | | |
| `0.5` | | | | |
| `1.0` | | | | |
| `1.5` | | | | |

---
## 실험 1-B. temperature=0.0 반복 3회

temperature=0.0에서는 정말 항상 같은 답이 나올까요?  
같은 설정으로 3번 호출해서 확인해봅니다.

> 💡 `temperature=0.0`은 `do_sample=False`(그리디 디코딩)이므로 이론상 동일한 결과가 나옵니다.

In [ ]:
print("─── temperature=0.0 반복 실험 (3회) ───")

responses_0 = []   # 3회 응답을 저장할 리스트

for i in range(3):   # 3번 반복
    result = generate_response(
        messages=[{"role": "user", "content": question}],
        temperature=0.0,     # 고정
        max_new_tokens=100,
    )
    answer = result["content"]
    responses_0.append(answer)   # 리스트에 저장
    print(f"\n[{i+1}회차]")
    print(answer[:120])   # 앞 120자만 출력
    tracker.add(result)

# 3회 응답이 모두 동일한지 확인
if responses_0[0] == responses_0[1] == responses_0[2]:
    print("\n✅ 3회 모두 동일한 응답! (temperature=0.0은 결정론적)")
else:
    print("\n⚠️ 약간의 차이 발생 (드물지만 가능)")

---
## 실험 1-C. max_new_tokens에 따른 응답 변화

`max_new_tokens`는 **"이 이상은 생성하지 마"** 라는 상한선입니다.  
너무 작으면 문장이 중간에 잘립니다. `finish_reason`이 `length`로 바뀌는 걸 확인하세요.

```
finish_reason: stop   → 모델이 자연스럽게 마무리 ✅
finish_reason: length → max_new_tokens에 걸려 강제 중단 ⚠️
```

> 💡 `finish_reason`은 `generate_response` 함수 내부에서  
> `출력 토큰 수 < max_new_tokens`이면 `"stop"`, 같으면 `"length"`로 판단합니다.

In [ ]:
# max_new_tokens를 30 → 100 → 300 순서로 바꿔가며 호출
q_long = "법원 판결문 작성 시 중요한 요소를 5가지 설명해 주세요."

for max_tok in [30, 100, 300]:
    result = generate_response(
        messages=[{"role": "user", "content": q_long}],
        temperature=0.3,         # 고정
        max_new_tokens=max_tok,  # ← 이 값만 바뀝니다
    )

    finish = result["finish_reason"]
    used   = result["output_tokens"]

    print(f"\n{'='*50}")
    print(f"[max_new_tokens={max_tok}]  실제 사용: {used}토큰  종료: {finish}")
    print(f"{'='*50}")
    print(result["content"])

    if finish == "length":   # 잘린 경우 경고
        print("\n⚠️ 문장이 중간에 잘렸습니다! max_new_tokens를 늘려야 합니다.")

    tracker.add(result)

---
## 실험 1-D. top_p 3단계 비교

`top_p`는 **단어 후보의 범위**를 정합니다.  
0.5면 확률 상위 50% 단어만, 1.0이면 모든 단어가 후보가 됩니다.

> 💡 실무에서는 `top_p=0.9`로 고정하고 `temperature`만 조절하는 게 일반적입니다.

In [ ]:
# top_p를 0.5 → 0.9 → 1.0 순서로 바꿔가며 호출
for p in [0.5, 0.9, 1.0]:
    result = generate_response(
        messages=[{"role": "user", "content": question}],
        temperature=0.7,     # 고정 (중간값)
        max_new_tokens=100,
        top_p=p,             # ← 이 값만 바뀝니다
    )

    print(f"\n[top_p={p}]")
    print(result["content"][:150])   # 앞 150자만 출력

    tracker.add(result)

---
## 실험 1-E. 환각(Hallucination) 관찰

temperature가 높으면 EXAONE이 **없는 법조문을 자신 있게 인용**하기도 합니다.  
이를 **환각(Hallucination)** 이라고 합니다.

아래 실험 후 인용된 법조문 번호를 검색해보세요. 실제로 존재하는지 확인해보는 것이 포인트입니다.

In [ ]:
# temperature=1.5에서 법조문 인용 정확도를 관찰합니다
result_high = generate_response(
    messages=[{"role": "user", "content": "임대차 보증금 반환 청구의 법적 근거를 구체적 법조문과 함께 설명해 주세요."}],
    temperature=1.5,     # 높은 값
    max_new_tokens=200,
)

print("[temperature=1.5] 법조문 인용 관찰")
print(result_high["content"])
print("\n⚠️ 위에서 인용된 법조문이 실제로 존재하는지 검색해보세요!")
print("   → 없는 조문을 자신 있게 인용하는 것이 '환각(Hallucination)'입니다.")

tracker.add(result_high)

---
## 📊 실험 결과 정리

모든 실험을 마쳤습니다. 아래 표를 직접 채워보세요.

### 태스크별 권장 파라미터 (내 판단)

| 태스크 | temperature | max_new_tokens | 이유 |
|--------|------------|----------------|------|
| 법률 Q&A | | | |
| JSON 추출 | | | |
| 합성 데이터 생성 | | | |
| 창의적 글쓰기 | | | |

### 💬 "같은 질문인데 답이 다르다" — 문제인가, 장점인가?

| 상황 | 문제 / 장점 | 이유 |
|------|-----------|------|
| 법률 Q&A 챗봇 | | |
| 아이디어 브레인스토밍 | | |
| 파인튜닝 합성 데이터 생성 | | |

In [ ]:
# 전체 실험 누적 토큰 확인
print("💰 오늘 실험 전체 토큰 사용량")
tracker.report()